![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Week 2, Day 3 -- Lab 2: Image Segmentation with a pretrained U-Net

Detection drew a **box** around each object. **Segmentation** is more precise:
it labels **every pixel** -- exactly which pixels are the object.

We'll use a **U-Net** (the classic segmentation model) that is **already trained**
to highlight abnormal regions in brain MRI scans. Reusing it is transfer learning.

## 1) Load a pretrained U-Net

In [ ]:
import torch

model = torch.hub.load("mateuszbuda/brain-segmentation-pytorch", "unet",
                       in_channels=3, out_channels=1, init_features=32, pretrained=True)
model.eval();

## 2) Get brain MRI scans (with ground-truth masks)

This downloads the dataset the model was trained on.

In [ ]:
import kagglehub, glob

path = kagglehub.dataset_download("mateuszbuda/lgg-mri-segmentation")
masks = sorted(glob.glob(path + "/**/*_mask.tif", recursive=True))

## 3) Pick 8 scans that actually contain a tumour

In [ ]:
import numpy as np
from PIL import Image

pairs = []
for m in masks:
    if np.array(Image.open(m)).max() > 0:           # keep only non-empty masks
        pairs.append((m.replace("_mask", ""), m))   # (image path, mask path)
    if len(pairs) == 8:
        break

## 4) A helper: run the U-Net on one image

It returns a **mask** -- True for every pixel the model marks as abnormal.

In [ ]:
from torchvision import transforms

def predict_mask(img):
    mean, std = np.mean(img, axis=(0, 1)), np.std(img, axis=(0, 1))
    x = transforms.Normalize(mean, std)(transforms.ToTensor()(img)).unsqueeze(0)
    with torch.no_grad():
        return (model(x)[0, 0] > 0.5).numpy()

## 5) Compare: image, prediction, ground truth, and overlay

For 8 scans we show the MRI, the U-Net's mask, the real mask, and the two combined.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(len(pairs), 4, figsize=(10, 2.5 * len(pairs)))
titles = ["MRI image", "predicted mask", "actual mask", "overlay"]
for r, (img_path, mask_path) in enumerate(pairs):
    img = Image.open(img_path).convert("RGB")
    pred = predict_mask(img)
    truth = np.array(Image.open(mask_path)) > 0
    ax[r, 0].imshow(img)
    ax[r, 1].imshow(pred, cmap="gray")
    ax[r, 2].imshow(truth, cmap="gray")
    ax[r, 3].imshow(img); ax[r, 3].imshow(pred, alpha=0.4, cmap="autumn")
    for c in range(4): ax[r, c].axis("off")
for c in range(4): ax[0, c].set_title(titles[c])
plt.tight_layout(); plt.show()

## Task: turn segmentation into a sticker maker

The same idea -- *find which pixels are the subject* -- powers **background
removal**. Swap the brain model for a general background remover and make your
own WhatsApp sticker. **Upload any photo** (a person, a pet, an object) and run
the cells below.

In [ ]:
%pip install rembg -q

In [ ]:
from rembg import remove
from google.colab import files

uploaded = files.upload()
name = next(iter(uploaded))
sticker = remove(Image.open(name))   # removes the background -> transparent cut-out
sticker.save("sticker.png")

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(sticker); plt.axis("off"); plt.title("your sticker"); plt.show()

## Recap

Three ways to understand an image, all using a **pretrained** model (transfer
learning): **classification** (*what*), **detection** (*what + where*, a box),
and **segmentation** (*which pixels*, a mask).

### *Contributed By: Sattam Altwaim*